# 📊 Financial Dashboard for Bloomberg BQuant

**功能包含：**
- 美歐殖利率曲線（Today/Month Start/Year Start）
- 10Y-2Y 利差計算 + Flatten/Steepen 指標
- 全球股市指數、美股板塊表現
- 外匯、商品、加密貨幣表格
- VIX、MOVE 波動率指標

---

In [ ]:
# 初始化套件
import bql
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

import bqplot as bqp
from bqplot import Figure, Axis, Lines, Bars, LinearScale, OrdinalScale
import bqwidgets as bqw
from IPython.display import display, HTML
import ipywidgets as widgets

# 初始化 BQL
bq = bql.Service()
print('✅ BQL Service initialized')

## 1️⃣ 美國國債殖利率曲線

In [ ]:
def get_us_treasury_yields():
    """獲取美國國債殖利率"""
    tickers = ['GB1 Govt', 'GB3 Govt', 'GB6 Govt', 'GB12 Govt',
               'GT2 Govt', 'GT3 Govt', 'GT5 Govt', 'GT7 Govt',
               'GT10 Govt', 'GT20 Govt', 'GT30 Govt']
    
    maturities = ['1M', '3M', '6M', '1Y', '2Y', '3Y', '5Y', '7Y', '10Y', '20Y', '30Y']
    maturity_values = [1/12, 3/12, 6/12, 1, 2, 3, 5, 7, 10, 20, 30]
    
    today = datetime.now()
    month_start = today.replace(day=1)
    year_start = today.replace(month=1, day=1)
    
    request = bql.Request(
        tickers,
        {
            'Today': bq.data.px_last(),
            'MonthStart': bq.data.px_last(dates=month_start.strftime('%Y-%m-%d')),
            'YearStart': bq.data.px_last(dates=year_start.strftime('%Y-%m-%d')),
        }
    )
    
    response = bq.execute(request)
    df = response[0].df()
    
    return {
        'maturities': maturities,
        'maturity_values': maturity_values,
        'today': df['Today'].tolist(),
        'month_start': df['MonthStart'].tolist(),
        'year_start': df['YearStart'].tolist(),
        'date_today': today.strftime('%Y-%m-%d'),
        'date_month_start': month_start.strftime('%Y-%m-%d'),
        'date_year_start': year_start.strftime('%Y-%m-%d'),
    }

# 獲取數據
us_yields = get_us_treasury_yields()
print(f"✅ US Treasury yields loaded as of {us_yields['date_today']}")

In [ ]:
# 繪製美國殖利率曲線
x_scale = LinearScale()
y_scale = LinearScale()

x_axis = Axis(scale=x_scale, label='Maturity (Years)', grid_lines='solid')
y_axis = Axis(scale=y_scale, orientation='vertical', label='Yield (%)', grid_lines='solid')

line_today = Lines(
    x=us_yields['maturity_values'],
    y=us_yields['today'],
    scales={'x': x_scale, 'y': y_scale},
    colors=['#2196F3'],
    labels=[f"Today ({us_yields['date_today']})"],
    display_legend=True,
    marker='circle'
)

line_month = Lines(
    x=us_yields['maturity_values'],
    y=us_yields['month_start'],
    scales={'x': x_scale, 'y': y_scale},
    colors=['#4CAF50'],
    labels=['Month Start'],
    display_legend=True,
    marker='square',
    line_style='dashed'
)

line_year = Lines(
    x=us_yields['maturity_values'],
    y=us_yields['year_start'],
    scales={'x': x_scale, 'y': y_scale},
    colors=['#F44336'],
    labels=['Year Start'],
    display_legend=True,
    marker='triangle-up',
    line_style='dotted'
)

fig_us_yield = Figure(
    marks=[line_today, line_month, line_year],
    axes=[x_axis, y_axis],
    title='🇺🇸 US Treasury Yield Curve',
    legend_location='top-right',
    layout=widgets.Layout(width='900px', height='400px')
)

display(fig_us_yield)

## 2️⃣ 歐洲（德國）國債殖利率曲線

In [ ]:
def get_euro_yields():
    """獲取歐洲（德國）國債殖利率"""
    tickers = ['GTDEM3M Govt', 'GTDEM6M Govt', 'GTDEM1Y Govt',
               'GTDEM2Y Govt', 'GTDEM3Y Govt', 'GTDEM5Y Govt',
               'GTDEM7Y Govt', 'GTDEM10Y Govt', 'GTDEM20Y Govt', 'GTDEM30Y Govt']
    
    maturities = ['3M', '6M', '1Y', '2Y', '3Y', '5Y', '7Y', '10Y', '20Y', '30Y']
    maturity_values = [3/12, 6/12, 1, 2, 3, 5, 7, 10, 20, 30]
    
    today = datetime.now()
    month_start = today.replace(day=1)
    year_start = today.replace(month=1, day=1)
    
    request = bql.Request(
        tickers,
        {
            'Today': bq.data.px_last(),
            'MonthStart': bq.data.px_last(dates=month_start.strftime('%Y-%m-%d')),
            'YearStart': bq.data.px_last(dates=year_start.strftime('%Y-%m-%d')),
        }
    )
    
    response = bq.execute(request)
    df = response[0].df()
    
    return {
        'maturities': maturities,
        'maturity_values': maturity_values,
        'today': df['Today'].tolist(),
        'month_start': df['MonthStart'].tolist(),
        'year_start': df['YearStart'].tolist(),
    }

euro_yields = get_euro_yields()
print('✅ Euro yields loaded')

In [ ]:
# 繪製歐洲殖利率曲線
x_scale_eu = LinearScale()
y_scale_eu = LinearScale()

x_axis_eu = Axis(scale=x_scale_eu, label='Maturity (Years)', grid_lines='solid')
y_axis_eu = Axis(scale=y_scale_eu, orientation='vertical', label='Yield (%)', grid_lines='solid')

line_today_eu = Lines(
    x=euro_yields['maturity_values'],
    y=euro_yields['today'],
    scales={'x': x_scale_eu, 'y': y_scale_eu},
    colors=['#2196F3'],
    labels=['Today'],
    display_legend=True,
    marker='circle'
)

line_month_eu = Lines(
    x=euro_yields['maturity_values'],
    y=euro_yields['month_start'],
    scales={'x': x_scale_eu, 'y': y_scale_eu},
    colors=['#4CAF50'],
    labels=['Month Start'],
    display_legend=True,
    marker='square',
    line_style='dashed'
)

line_year_eu = Lines(
    x=euro_yields['maturity_values'],
    y=euro_yields['year_start'],
    scales={'x': x_scale_eu, 'y': y_scale_eu},
    colors=['#F44336'],
    labels=['Year Start'],
    display_legend=True,
    marker='triangle-up',
    line_style='dotted'
)

fig_eu_yield = Figure(
    marks=[line_today_eu, line_month_eu, line_year_eu],
    axes=[x_axis_eu, y_axis_eu],
    title='🇪🇺 Euro Area Yield Curve (Germany)',
    legend_location='top-right',
    layout=widgets.Layout(width='900px', height='400px')
)

display(fig_eu_yield)

## 3️⃣ 10Y-2Y 利差分析 + Flatten/Steepen 指標

In [ ]:
def analyze_spread(yields_data):
    """分析 10Y-2Y 利差"""
    idx_2y = yields_data['maturities'].index('2Y')
    idx_10y = yields_data['maturities'].index('10Y')
    
    spread_today = (yields_data['today'][idx_10y] - yields_data['today'][idx_2y]) * 100
    spread_month = (yields_data['month_start'][idx_10y] - yields_data['month_start'][idx_2y]) * 100
    spread_year = (yields_data['year_start'][idx_10y] - yields_data['year_start'][idx_2y]) * 100
    
    spread_change = spread_today - spread_month
    
    if spread_change > 5:
        trend = 'Steepening 📈'
        trend_color = '#4CAF50'
    elif spread_change < -5:
        trend = 'Flattening 📉'
        trend_color = '#F44336'
    else:
        trend = 'Stable ➡️'
        trend_color = '#9E9E9E'
    
    return {
        'yield_2y': yields_data['today'][idx_2y],
        'yield_10y': yields_data['today'][idx_10y],
        'spread_today': spread_today,
        'spread_month': spread_month,
        'spread_year': spread_year,
        'spread_change': spread_change,
        'trend': trend,
        'trend_color': trend_color,
    }

spread_data = analyze_spread(us_yields)

# 顯示利差分析
spread_html = f"""
<div style="background: linear-gradient(135deg, #f5f7fa 0%, #c3cfe2 100%); 
            padding: 20px; border-radius: 12px; margin: 10px 0;">
    <h2 style="margin-top: 0;">📊 US 10Y-2Y Spread Analysis</h2>
    <div style="display: flex; gap: 40px;">
        <div>
            <h3>Current Yields</h3>
            <p><b>2Y Yield:</b> {spread_data['yield_2y']:.2f}%</p>
            <p><b>10Y Yield:</b> {spread_data['yield_10y']:.2f}%</p>
        </div>
        <div>
            <h3>Spread (bps)</h3>
            <p><b>Today:</b> {spread_data['spread_today']:.0f}</p>
            <p><b>Month Ago:</b> {spread_data['spread_month']:.0f}</p>
            <p><b>Year Ago:</b> {spread_data['spread_year']:.0f}</p>
        </div>
        <div>
            <h3>Trend</h3>
            <p style="font-size: 24px; color: {spread_data['trend_color']}; font-weight: bold;">
                {spread_data['trend']}
            </p>
            <p>Monthly Change: <span style="color: {spread_data['trend_color']};">
                {spread_data['spread_change']:+.0f} bps</span></p>
        </div>
    </div>
</div>
"""

display(HTML(spread_html))

In [ ]:
# 利差長條圖
x_scale_spread = OrdinalScale()
y_scale_spread = LinearScale()

x_axis_spread = Axis(scale=x_scale_spread, grid_lines='none')
y_axis_spread = Axis(scale=y_scale_spread, orientation='vertical', label='Spread (bps)', grid_lines='solid')

colors_spread = ['#9E9E9E', '#2196F3', spread_data['trend_color']]

bars_spread = Bars(
    x=['Year Start', 'Month Start', 'Today'],
    y=[spread_data['spread_year'], spread_data['spread_month'], spread_data['spread_today']],
    scales={'x': x_scale_spread, 'y': y_scale_spread},
    colors=colors_spread,
    padding=0.4
)

fig_spread = Figure(
    marks=[bars_spread],
    axes=[x_axis_spread, y_axis_spread],
    title=f'US 10Y-2Y Spread: {spread_data["trend"]}',
    layout=widgets.Layout(width='600px', height='350px')
)

display(fig_spread)

## 4️⃣ 全球股市指數

In [ ]:
def get_global_indices():
    """獲取全球股市指數"""
    indices = {
        'S&P 500': 'SPX Index',
        'NASDAQ': 'CCMP Index',
        'Dow Jones': 'INDU Index',
        'Russell 2000': 'RTY Index',
        'STOXX 600': 'SXXP Index',
        'DAX': 'DAX Index',
        'FTSE 100': 'UKX Index',
        'CAC 40': 'CAC Index',
        'Nikkei 225': 'NKY Index',
        'Hang Seng': 'HSI Index',
        'Shanghai Comp': 'SHCOMP Index',
        'TAIEX': 'TWSE Index',
        'KOSPI': 'KOSPI Index',
    }
    
    tickers = list(indices.values())
    names = list(indices.keys())
    year_start = datetime.now().replace(month=1, day=1)
    
    request = bql.Request(
        tickers,
        {
            'Price': bq.data.px_last(),
            'Change': bq.data.day_to_day_total_return(),
            'YTD_Start': bq.data.px_last(dates=year_start.strftime('%Y-%m-%d')),
        }
    )
    
    response = bq.execute(request)
    df = response[0].df()
    df['YTD'] = ((df['Price'] - df['YTD_Start']) / df['YTD_Start'] * 100)
    df['Name'] = names
    
    return df[['Name', 'Price', 'Change', 'YTD']]

indices_df = get_global_indices()
print('✅ Global indices loaded')

In [ ]:
# 顯示全球指數表格
def format_change(val):
    color = '#4CAF50' if val >= 0 else '#F44336'
    return f'<span style="color:{color};">{val:+.2f}%</span>'

indices_html = """
<style>
    .indices-table { width: 100%; border-collapse: collapse; }
    .indices-table th { background: #667eea; color: white; padding: 12px; text-align: left; }
    .indices-table td { padding: 10px; border-bottom: 1px solid #ddd; }
    .indices-table tr:hover { background: #f5f5f5; }
</style>
<h2>🌍 Global Market Indices</h2>
<table class="indices-table">
    <tr><th>Index</th><th style="text-align:right;">Price</th><th style="text-align:right;">Daily</th><th style="text-align:right;">YTD</th></tr>
"""

for _, row in indices_df.iterrows():
    daily_color = '#4CAF50' if row['Change'] >= 0 else '#F44336'
    ytd_color = '#4CAF50' if row['YTD'] >= 0 else '#F44336'
    indices_html += f"""
    <tr>
        <td><b>{row['Name']}</b></td>
        <td style="text-align:right;">{row['Price']:,.2f}</td>
        <td style="text-align:right; color:{daily_color};">{row['Change']:+.2f}%</td>
        <td style="text-align:right; color:{ytd_color};">{row['YTD']:+.1f}%</td>
    </tr>
    """

indices_html += "</table>"
display(HTML(indices_html))

## 5️⃣ 美股板塊表現（雙向長條圖）

In [ ]:
def get_us_sectors():
    """獲取美股板塊 ETF 表現"""
    sectors = {
        'Technology': 'XLK US Equity',
        'Healthcare': 'XLV US Equity',
        'Financials': 'XLF US Equity',
        'Consumer Disc.': 'XLY US Equity',
        'Industrials': 'XLI US Equity',
        'Energy': 'XLE US Equity',
        'Materials': 'XLB US Equity',
        'Utilities': 'XLU US Equity',
        'Real Estate': 'XLRE US Equity',
        'Comm. Services': 'XLC US Equity',
        'Consumer Staples': 'XLP US Equity',
    }
    
    tickers = list(sectors.values())
    names = list(sectors.keys())
    
    year_start = datetime.now().replace(month=1, day=1)
    
    request = bql.Request(
        tickers,
        {
            'Price': bq.data.px_last(),
            'Daily': bq.data.day_to_day_total_return(),
            'YearStart': bq.data.px_last(dates=year_start.strftime('%Y-%m-%d')),
        }
    )
    
    response = bq.execute(request)
    df = response[0].df()
    df['YTD'] = ((df['Price'] - df['YearStart']) / df['YearStart'] * 100)
    df['Name'] = names
    
    return df.sort_values('Daily', ascending=False)

sectors_df = get_us_sectors()
print('✅ US sector data loaded')

In [ ]:
# 板塊雙向長條圖
sector_names = sectors_df['Name'].tolist()
daily_changes = sectors_df['Daily'].tolist()

colors_sectors = ['#4CAF50' if x >= 0 else '#F44336' for x in daily_changes]

x_scale_sec = LinearScale()
y_scale_sec = OrdinalScale()

x_axis_sec = Axis(scale=x_scale_sec, label='Daily Change (%)', grid_lines='solid')
y_axis_sec = Axis(scale=y_scale_sec, orientation='vertical', grid_lines='none')

bars_sectors = Bars(
    x=daily_changes,
    y=sector_names,
    scales={'x': x_scale_sec, 'y': y_scale_sec},
    colors=colors_sectors,
    orientation='horizontal',
    padding=0.2
)

fig_sectors = Figure(
    marks=[bars_sectors],
    axes=[x_axis_sec, y_axis_sec],
    title='🏢 US Sector Performance (Daily Change)',
    layout=widgets.Layout(width='800px', height='450px'),
    fig_margin={'top': 60, 'bottom': 60, 'left': 140, 'right': 60}
)

display(fig_sectors)

## 6️⃣ 外匯、商品數據

In [ ]:
def get_forex_commodities():
    """獲取外匯和商品數據"""
    
    # 外匯
    fx_tickers = {
        'EUR/USD': 'EURUSD Curncy',
        'USD/JPY': 'USDJPY Curncy',
        'GBP/USD': 'GBPUSD Curncy',
        'USD/CNY': 'USDCNY Curncy',
        'USD/TWD': 'USDTWD Curncy',
        'DXY': 'DXY Curncy',
    }
    
    # 商品
    comm_tickers = {
        'Gold': 'GC1 Comdty',
        'Silver': 'SI1 Comdty',
        'WTI Crude': 'CL1 Comdty',
        'Brent Crude': 'CO1 Comdty',
        'Natural Gas': 'NG1 Comdty',
        'Copper': 'HG1 Comdty',
    }
    
    all_tickers = list(fx_tickers.values()) + list(comm_tickers.values())
    all_names = list(fx_tickers.keys()) + list(comm_tickers.keys())
    
    week_ago = datetime.now() - timedelta(days=7)
    
    request = bql.Request(
        all_tickers,
        {
            'Price': bq.data.px_last(),
            'Change': bq.data.day_to_day_total_return(),
            'WeekAgo': bq.data.px_last(dates=week_ago.strftime('%Y-%m-%d')),
        }
    )
    
    response = bq.execute(request)
    df = response[0].df()
    df['Weekly'] = ((df['Price'] - df['WeekAgo']) / df['WeekAgo'] * 100)
    df['Name'] = all_names
    
    fx_df = df.head(len(fx_tickers))
    comm_df = df.tail(len(comm_tickers))
    
    return fx_df, comm_df

fx_df, comm_df = get_forex_commodities()
print('✅ FX and Commodities data loaded')

In [ ]:
# 顯示外匯和商品表格
def create_table_html(df, title):
    html = f"""
    <div style="flex: 1; margin: 10px;">
        <h3>{title}</h3>
        <table style="width:100%; border-collapse:collapse;">
            <tr style="background:#667eea; color:white;">
                <th style="padding:10px; text-align:left;">Name</th>
                <th style="padding:10px; text-align:right;">Price</th>
                <th style="padding:10px; text-align:right;">Daily</th>
                <th style="padding:10px; text-align:right;">Weekly</th>
            </tr>
    """
    
    for _, row in df.iterrows():
        daily_color = '#4CAF50' if row['Change'] >= 0 else '#F44336'
        weekly_color = '#4CAF50' if row['Weekly'] >= 0 else '#F44336'
        html += f"""
        <tr style="border-bottom:1px solid #ddd;">
            <td style="padding:8px;"><b>{row['Name']}</b></td>
            <td style="padding:8px; text-align:right;">{row['Price']:.4f}</td>
            <td style="padding:8px; text-align:right; color:{daily_color};">{row['Change']:+.2f}%</td>
            <td style="padding:8px; text-align:right; color:{weekly_color};">{row['Weekly']:+.2f}%</td>
        </tr>
        """
    
    html += "</table></div>"
    return html

combined_html = f"""
<div style="display:flex; flex-wrap:wrap;">
    {create_table_html(fx_df, '💱 Foreign Exchange')}
    {create_table_html(comm_df, '🛢️ Commodities')}
</div>
"""

display(HTML(combined_html))

## 7️⃣ 波動率指標 (VIX, MOVE)

In [ ]:
def get_volatility():
    """獲取波動率指標"""
    tickers = ['VIX Index', 'MOVE Index']
    names = ['VIX', 'MOVE']
    
    yesterday = datetime.now() - timedelta(days=1)
    
    request = bql.Request(
        tickers,
        {
            'Value': bq.data.px_last(),
            'Yesterday': bq.data.px_last(dates=yesterday.strftime('%Y-%m-%d')),
        }
    )
    
    response = bq.execute(request)
    df = response[0].df()
    df['Change'] = ((df['Value'] - df['Yesterday']) / df['Yesterday'] * 100)
    df['Name'] = names
    
    return df

vol_df = get_volatility()
print('✅ Volatility data loaded')

In [ ]:
# 顯示波動率指標
def get_vix_level(value):
    if value < 15:
        return 'Low 🟢', '#4CAF50'
    elif value < 25:
        return 'Normal 🟡', '#FFC107'
    else:
        return 'High 🔴', '#F44336'

def get_move_level(value):
    if value < 80:
        return 'Low 🟢', '#4CAF50'
    elif value < 120:
        return 'Normal 🟡', '#FFC107'
    else:
        return 'High 🔴', '#F44336'

vix_val = vol_df[vol_df['Name'] == 'VIX']['Value'].values[0]
vix_chg = vol_df[vol_df['Name'] == 'VIX']['Change'].values[0]
move_val = vol_df[vol_df['Name'] == 'MOVE']['Value'].values[0]
move_chg = vol_df[vol_df['Name'] == 'MOVE']['Change'].values[0]

vix_level, vix_color = get_vix_level(vix_val)
move_level, move_color = get_move_level(move_val)

vol_html = f"""
<h2>📊 Volatility Indicators</h2>
<div style="display:flex; gap:30px; justify-content:center;">
    <div style="background:linear-gradient(135deg, #f8f9fa 0%, #e9ecef 100%);
                border-radius:16px; padding:25px; text-align:center; min-width:200px;
                box-shadow: 0 4px 12px rgba(0,0,0,0.1);">
        <div style="font-size:16px; color:#666;">VIX Index</div>
        <div style="font-size:42px; font-weight:bold; margin:10px 0;">{vix_val:.1f}</div>
        <div style="background:{vix_color}; color:white; padding:6px 16px;
                    border-radius:20px; display:inline-block;">{vix_level}</div>
        <div style="margin-top:10px; color:{'#4CAF50' if vix_chg < 0 else '#F44336'};">
            Change: {vix_chg:+.1f}%
        </div>
    </div>
    <div style="background:linear-gradient(135deg, #f8f9fa 0%, #e9ecef 100%);
                border-radius:16px; padding:25px; text-align:center; min-width:200px;
                box-shadow: 0 4px 12px rgba(0,0,0,0.1);">
        <div style="font-size:16px; color:#666;">MOVE Index</div>
        <div style="font-size:42px; font-weight:bold; margin:10px 0;">{move_val:.1f}</div>
        <div style="background:{move_color}; color:white; padding:6px 16px;
                    border-radius:20px; display:inline-block;">{move_level}</div>
        <div style="margin-top:10px; color:{'#4CAF50' if move_chg < 0 else '#F44336'};">
            Change: {move_chg:+.1f}%
        </div>
    </div>
</div>
"""

display(HTML(vol_html))

## 8️⃣ 加密貨幣

In [ ]:
def get_crypto():
    """獲取加密貨幣數據"""
    tickers = ['XBTUSD BGN Curncy', 'XETUSD BGN Curncy']
    names = ['Bitcoin', 'Ethereum']
    
    week_ago = datetime.now() - timedelta(days=7)
    
    request = bql.Request(
        tickers,
        {
            'Price': bq.data.px_last(),
            'Change': bq.data.day_to_day_total_return(),
            'WeekAgo': bq.data.px_last(dates=week_ago.strftime('%Y-%m-%d')),
        }
    )
    
    response = bq.execute(request)
    df = response[0].df()
    df['Weekly'] = ((df['Price'] - df['WeekAgo']) / df['WeekAgo'] * 100)
    df['Name'] = names
    
    return df

try:
    crypto_df = get_crypto()
    
    crypto_html = """
    <h2>🪙 Cryptocurrency</h2>
    <table style="width:100%; max-width:600px; border-collapse:collapse;">
        <tr style="background:#667eea; color:white;">
            <th style="padding:12px; text-align:left;">Crypto</th>
            <th style="padding:12px; text-align:right;">Price</th>
            <th style="padding:12px; text-align:right;">24h</th>
            <th style="padding:12px; text-align:right;">7d</th>
        </tr>
    """
    
    for _, row in crypto_df.iterrows():
        daily_color = '#4CAF50' if row['Change'] >= 0 else '#F44336'
        weekly_color = '#4CAF50' if row['Weekly'] >= 0 else '#F44336'
        crypto_html += f"""
        <tr style="border-bottom:1px solid #ddd;">
            <td style="padding:10px;"><b>{row['Name']}</b></td>
            <td style="padding:10px; text-align:right;">${row['Price']:,.2f}</td>
            <td style="padding:10px; text-align:right; color:{daily_color};">{row['Change']:+.2f}%</td>
            <td style="padding:10px; text-align:right; color:{weekly_color};">{row['Weekly']:+.2f}%</td>
        </tr>
        """
    
    crypto_html += "</table>"
    display(HTML(crypto_html))
    print('✅ Crypto data loaded')
except Exception as e:
    print(f'⚠️ Could not load crypto data: {e}')

---

## 📋 Dashboard Summary

This dashboard provides:

1. **Yield Curves** - US and Euro area treasury yield curves with historical comparison
2. **Spread Analysis** - 10Y-2Y spread with flattening/steepening indicator
3. **Global Indices** - Major market indices performance
4. **US Sectors** - S&P 500 sector ETF performance
5. **FX & Commodities** - Major currency pairs and commodity prices
6. **Volatility** - VIX and MOVE index levels
7. **Crypto** - Bitcoin and Ethereum prices

---

*Generated using Bloomberg BQuant with BQL*